# Graphs — Subtopic 5: Cycle Detection & Bipartite

**Kernel:** C++17 (xeus-cling, `xcpp17`)

Pedagogical order — cleanest baseline first, then variants that stress one new idea each:
1. Cycle Detection in Undirected Graph (BFS) — parent tracking
2. Cycle Detection in Undirected Graph (DFS) — same problem, DFS framing
3. Cycle Detection in Directed Graph (DFS) — three-color derivation, why parent tracking fails here
4. Bipartite Graph (BFS) — 2-coloring via layer parity
5. Bipartite Graph (DFS) — same problem, DFS framing
6. Comparison table — which technique for which cycle-detection variant, and why


## 5.1 Cycle Detection in Undirected Graph (BFS)

### State definition
$\text{visited}[v] \in \{\text{false}, \text{true}\}$, and $\text{parent}[v]$ = the vertex from
which $v$ was first discovered during the BFS. A **cycle exists** iff, while scanning $u$'s
neighbors, some neighbor $w$ is found with $\text{visited}[w] = \text{true}$ **and** $w \neq
\text{parent}[u]$.

### Invariant
At the moment $u$ is dequeued, every vertex already visited is either an ancestor of $u$ in the
BFS tree (reachable via the tree-edge path back through parents) or belongs to a different,
already-fully-explored branch. The parent pointer records the *one* legitimate reason a visited
neighbor can appear: the edge you just came from.

### Why it works — proof of the "already visited and not parent" check

**Claim:** in an undirected graph, encountering a visited neighbor $w \neq \text{parent}[u]$ while
processing $u$ implies a cycle exists.

*Proof:* the edge $(u,w)$ exists (we are examining it). Since $w$ is already visited and $w \neq
\text{parent}[u]$, $w$ was discovered by some other path not using the edge $(u,w)$ — meaning there
is a path $w \rightsquigarrow \cdots \rightsquigarrow u$ in the BFS-explored graph not using the
edge $(w,u)$ directly. Combined with the edge $(u,w)$ itself, this closes a cycle: path from $w$ to
$u$, plus edge back to $w$. Conversely, if the graph is acyclic (a forest), every non-tree
adjacency check during BFS is either an unvisited neighbor (tree edge) or exactly the parent edge
(each undirected edge is examined from both endpoints — Subtopic 2.2's proof that undirected DFS
has only tree/back edges applies identically to BFS's adjacency scanning) — so the "visited and not
parent" condition never fires on a forest. $\blacksquare$

**Why "not parent" is essential:** every undirected edge $(u, \text{parent}[u])$ is examined
*twice* — once when parent discovered $u$, and once when $u$ scans back toward parent. Without
excluding the parent, this single tree edge would be misreported as a cycle on every graph,
including simple trees with zero actual cycles.

### Boundary transitions table

| Decision point | Condition | Action |
|---|---|---|
| Neighbor unvisited | $\text{visited}[w] = \text{false}$ | tree edge — visit, set $\text{parent}[w] = u$, enqueue |
| Neighbor visited, is parent | $w = \text{parent}[u]$ | the edge just traversed backward — not a cycle, skip |
| Neighbor visited, not parent | $w \neq \text{parent}[u]$ | cycle found — report true |
| Self-loop | $w = u$ | trivially $w \neq \text{parent}[u]$ (parent can't equal self unless self-seeded) — correctly flags as a cycle |

### The delta
The non-obvious insight: parent tracking is a **single extra integer per vertex**, but it is the
entire mechanism that distinguishes "this is the tree edge I arrived on" from "this is a genuine
back-connection forming a cycle" — undirected graphs need this because every edge is inherently
bidirectional, so naive "have I seen this neighbor" logic would misfire on every tree edge.


In [ ]:
// Cycle Detection in Undirected Graph (BFS): parent tracking distinguishes tree-edge backtrack from real cycle
#include <iostream>
#include <vector>
#include <queue>
using namespace std;
#define vi vector<int>
#define vvi vector<vector<int>>
#define vb vector<bool>

// bfsHasCycleFrom: BFS from src, using parent[] to filter out the trivial "walked back along the tree edge" case.
bool bfsHasCycleFrom(int src, vvi& adj, vb& visited) {
    queue<int> q;
    vi parent(adj.size(), -1);
    visited[src] = true;
    q.push(src);
    while (!q.empty()) {
        int u = q.front(); q.pop();
        for (int w : adj[u]) {
            if (!visited[w]) {
                visited[w] = true;
                parent[w] = u;                  // record WHY w is visited -- the tree edge we arrived on
                q.push(w);
            } else if (w != parent[u]) {
                return true;                     // visited AND not the parent edge -> genuine cycle
            }
        }
    }
    return false;
}

// hasCycleUndirected: runs BFS from every unvisited component root (graph may be disconnected).
bool hasCycleUndirected(int V, vvi& adj) {
    vb visited(V, false);
    for (int v = 0; v < V; v++) {
        if (!visited[v] && bfsHasCycleFrom(v, adj, visited)) return true;
    }
    return false;
}


In [ ]:
// Test cell: input -> actual (Expected: X)

// Triangle: 0-1-2-0, a genuine cycle
{
    vvi adj(3); adj[0]={1,2}; adj[1]={0,2}; adj[2]={0,1};
    cout << "hasCycleUndirected(triangle) -> " << hasCycleUndirected(3, adj) << " (Expected: 1)\n";
}

// Simple tree: 0-1, 1-2, 1-3, no cycle
{
    vvi adj(4); adj[0]={1}; adj[1]={0,2,3}; adj[2]={1}; adj[3]={1};
    cout << "hasCycleUndirected(tree) -> " << hasCycleUndirected(4, adj) << " (Expected: 0)\n";
}

// Single node, no edges
{
    vvi adj(1);
    cout << "hasCycleUndirected(single node) -> " << hasCycleUndirected(1, adj) << " (Expected: 0)\n";
}

// Single edge, no cycle
{
    vvi adj(2); adj[0]={1}; adj[1]={0};
    cout << "hasCycleUndirected(single edge) -> " << hasCycleUndirected(2, adj) << " (Expected: 0)\n";
}

// Disconnected: one tree component, one cyclic component
{
    vvi adj(6);
    adj[0]={1}; adj[1]={0};                          // tree component
    adj[2]={3,4}; adj[3]={2,4}; adj[4]={2,3}; adj[5]={};  // triangle {2,3,4} + isolated 5
    cout << "hasCycleUndirected(disconnected, one cyclic) -> " << hasCycleUndirected(6, adj) << " (Expected: 1)\n";
}

// Path graph (acyclic, longer)
{
    vvi adj(5);
    for (int i=0;i<4;i++){adj[i].push_back(i+1); adj[i+1].push_back(i);}
    cout << "hasCycleUndirected(path) -> " << hasCycleUndirected(5, adj) << " (Expected: 0)\n";
}


## 5.2 Cycle Detection in Undirected Graph (DFS)

### State definition
Same as 5.1: $\text{visited}[v]$, and a `parent` argument threaded through the recursion instead of
a stored array (though a stored array works identically).

### Invariant
Identical logical invariant to 5.1 — the only change is *which traversal order* discovers vertices.
This is the direct DFS analogue: by Subtopic 2.2's proof that **undirected DFS has only tree and
back edges** (never forward/cross), every non-tree adjacency encountered during DFS is a back
edge to some ancestor — and a back edge to any ancestor other than the immediate parent
unambiguously signals a cycle, for exactly the same reason as 5.1.

### Why it works
Direct corollary of Subtopic 2.2's edge classification: on an undirected graph, a "visited, not
parent" neighbor is *always* a back edge to a strict ancestor (never a cross edge, since cross
edges cannot occur in undirected DFS) — so finding one is both necessary and sufficient evidence
of a cycle, matching 5.1's BFS proof exactly, just via recursion-stack ancestry (GRAY-equivalent)
instead of BFS-layer ancestry.

### Boundary transitions table

Identical to 5.1's table — DFS and BFS differ only in *how* they discover the tree structure, not
in *what counts as a cycle signal*.

### The delta
The non-obvious insight (this is the promised BFS-vs-DFS comparison for this variant): **for
undirected cycle detection, BFS and DFS are equally natural** — both rely on the exact same
"visited and not parent" rule, because undirected graphs' lack of cross/forward edges (proved in
Subtopic 2.2) means neither traversal order introduces any extra edge case the other doesn't have.
This is in sharp contrast to the *directed* case (5.3), where DFS is essential and BFS-with-parent
genuinely fails.


In [ ]:
// Cycle Detection in Undirected Graph (DFS): identical logic to 5.1, recursion-stack framing
#include <iostream>
#include <vector>
using namespace std;
#define vi vector<int>
#define vvi vector<vector<int>>
#define vb vector<bool>

// dfsHasCycle: parent threaded as a recursion parameter -- same "visited and not parent" rule as BFS.
bool dfsHasCycle(int u, int parent, vvi& adj, vb& visited) {
    visited[u] = true;
    for (int w : adj[u]) {
        if (!visited[w]) {
            if (dfsHasCycle(w, u, adj, visited)) return true;   // tree edge: recurse, propagate any cycle found deeper
        } else if (w != parent) {
            return true;                                         // visited, not parent -> back edge to a non-parent ancestor
        }
    }
    return false;
}

bool hasCycleUndirectedDFS(int V, vvi& adj) {
    vb visited(V, false);
    for (int v = 0; v < V; v++) {
        if (!visited[v] && dfsHasCycle(v, -1, adj, visited)) return true;   // -1 = no parent (root of this component)
    }
    return false;
}


In [ ]:
// Test cell: input -> actual (Expected: X)

// Triangle
{
    vvi adj(3); adj[0]={1,2}; adj[1]={0,2}; adj[2]={0,1};
    cout << "hasCycleUndirectedDFS(triangle) -> " << hasCycleUndirectedDFS(3, adj) << " (Expected: 1)\n";
}

// Tree
{
    vvi adj(4); adj[0]={1}; adj[1]={0,2,3}; adj[2]={1}; adj[3]={1};
    cout << "hasCycleUndirectedDFS(tree) -> " << hasCycleUndirectedDFS(4, adj) << " (Expected: 0)\n";
}

// Single node
{
    vvi adj(1);
    cout << "hasCycleUndirectedDFS(single node) -> " << hasCycleUndirectedDFS(1, adj) << " (Expected: 0)\n";
}

// Self-loop -- edge (0,0) with parent -1 -- neighbor 0 is visited and != parent(-1) -> cycle
{
    vvi adj(1); adj[0] = {0};
    cout << "hasCycleUndirectedDFS(self-loop) -> " << hasCycleUndirectedDFS(1, adj) << " (Expected: 1)\n";
}

// Fully connected K4 -- definitely cyclic
{
    vvi adj(4);
    for (int i=0;i<4;i++) for (int j=0;j<4;j++) if (i!=j) adj[i].push_back(j);
    cout << "hasCycleUndirectedDFS(K4) -> " << hasCycleUndirectedDFS(4, adj) << " (Expected: 1)\n";
}

// Cross-check against 5.1's BFS result on the same disconnected mixed graph
{
    vvi adj(6);
    adj[0]={1}; adj[1]={0};
    adj[2]={3,4}; adj[3]={2,4}; adj[4]={2,3}; adj[5]={};
    cout << "hasCycleUndirectedDFS(disconnected, one cyclic) -> " << hasCycleUndirectedDFS(6, adj) << " (Expected: 1, matches BFS result from 5.1)\n";
}


## 5.3 Cycle Detection in Directed Graph (DFS)

### State definition — three-color derivation from first principles
$\text{color}[v] \in \{W, G, B\}$, exactly as defined in Subtopic 2.2:
$\text{color}[v] = G$ **iff** $v$ is currently on the DFS recursion stack (an active ancestor of
the call in progress).

### Invariant — why parent tracking fails here, and why GRAY is the correct replacement

**Why the undirected "not parent" trick fails on directed graphs.** In an undirected graph, every
edge is bidirectional, so the *only* way to see an already-visited neighbor via a legitimate tree
edge is by walking back along that exact edge — hence excluding "the parent" suffices. In a
**directed** graph, edges are one-way: a vertex $u$ can have a directed edge to an ancestor $a$
that is **not** its immediate parent (e.g. $a \to x \to u \to a$, a 3-cycle where $u$'s parent is
$x$, not $a$) — "not equal to parent" would wrongly clear this as safe, since $a \neq
\text{parent}[u]$ yet $(u,a)$ closes a real cycle. Worse, a directed graph can also have edges to
an **already-fully-finished** vertex that is *not* an ancestor at all (a cross/forward edge, per
Subtopic 2.2) — this is completely safe and must **not** be flagged, but "visited" alone (without
distinguishing GRAY from BLACK) cannot tell the two apart.

**Why GRAY is exactly the right test.** By the recursion-stack invariant (Subtopic 2.2, restated
here as the crux of this problem): $\text{color}[v] = G \iff v$ is an ancestor of the current call.
An edge $(u,v)$ with $\text{color}[v] = G$ is therefore, by definition, an edge from the current
vertex back to one of its own active ancestors — exactly the structural definition of a directed
cycle (there is a path from $v$ down the recursion stack to $u$, plus the edge $u \to v$ closing
the loop). An edge to a BLACK vertex, by contrast, points to a vertex that has already finished —
by the invariant, a finished vertex is provably *not* on the current path, so no cycle is closed by
that edge, regardless of any other relationship between $u$ and $v$ in the graph.

**Formal proof that GRAY $\Rightarrow$ cycle exists:** if $\text{color}[v] = G$ when edge $(u,v)$
is examined, then $v$ is on the call stack, meaning there is an active chain of recursive calls
$v = c_0 \to c_1 \to \cdots \to c_k = u$ where each $c_i \to c_{i+1}$ is a tree edge (this is what
"on the call stack" means operationally). Each tree edge is a real graph edge, so
$v \to c_1 \to \cdots \to u$ is a real directed path in $G$. Combined with the edge $(u,v)$ just
examined, this is a directed cycle. $\blacksquare$ Conversely, if the graph has a directed cycle
$v_0 \to v_1 \to \cdots \to v_0$, consider the first $v_i$ visited by DFS: by the time DFS explores
edge $v_{i-1} \to v_i$ (or wraps around to $v_0$), all other cycle vertices are either undiscovered
(will become descendants when reached from $v_i$) or currently GRAY — eventually the edge closing
the cycle back to $v_i$ is examined while $v_i$ is still GRAY (it cannot have finished, since one of
its own descendants, the rest of the cycle, is still being explored), so the algorithm is
guaranteed to detect it.

### Boundary transitions table

| Decision point | Condition | Action |
|---|---|---|
| Neighbor WHITE | $\text{color}[v] = W$ | tree edge — recurse |
| Neighbor GRAY | $\text{color}[v] = G$ | back edge to an active ancestor — cycle found, return true |
| Neighbor BLACK | $\text{color}[v] = B$ | forward or cross edge — provably safe, no cycle from this edge |
| On call exit | all neighbors processed, no cycle found through $u$ | set $\text{color}[u] = B$ |

### The delta
The non-obvious insight, stated precisely: **undirected cycle detection needs parent tracking;
directed cycle detection needs recursion-stack (GRAY) tracking** — and the reason is that
"already visited" collapses two structurally different situations in a directed graph (active
ancestor vs. finished, unrelated-or-descendant vertex) that must be told apart, whereas in an
undirected graph those two situations coincide (Subtopic 2.2's proof that undirected DFS never
produces forward/cross edges means "visited" and "GRAY-or-was-an-ancestor" are effectively the same
information there). The naive undirected approach — "visited and not parent" — silently miscounts
directed cross/forward edges as safe only by accident (parent exclusion doesn't address them at
all); it is the three-color scheme, not parent-exclusion, that is the real fix, and it happens to
subsume the undirected case correctly too (GRAY-check works for undirected graphs as well, since
GRAY there is exactly "on the current tree path," which for undirected graphs collapses to
"not-yet-finished ancestor" — but simpler parent-tracking suffices there, so it's conventionally
used for clarity).


In [ ]:
// Cycle Detection in Directed Graph (DFS): WHITE/GRAY/BLACK three-color scheme
#include <iostream>
#include <vector>
using namespace std;
#define vi vector<int>
#define vvi vector<vector<int>>

enum Color { WHITE, GRAY, BLACK };

// dfsHasCycleDirected: GRAY neighbor = active ancestor = cycle. BLACK neighbor = finished, safe.
bool dfsHasCycleDirected(int u, vvi& adj, vector<Color>& color) {
    color[u] = GRAY;                      // enters recursion stack
    for (int v : adj[u]) {
        if (color[v] == WHITE) {
            if (dfsHasCycleDirected(v, adj, color)) return true;   // tree edge, propagate any cycle found deeper
        } else if (color[v] == GRAY) {
            return true;                   // back edge to an ACTIVE ancestor -- genuine directed cycle
        }
        // color[v] == BLACK: forward or cross edge -- provably safe (proof above), do nothing
    }
    color[u] = BLACK;                      // leaves recursion stack, fully finished
    return false;
}

bool hasCycleDirected(int V, vvi& adj) {
    vector<Color> color(V, WHITE);
    for (int v = 0; v < V; v++) {
        if (color[v] == WHITE && dfsHasCycleDirected(v, adj, color)) return true;
    }
    return false;
}


In [ ]:
// Test cell: input -> actual (Expected: X)

// Directed 3-cycle: 0->1->2->0
{
    vvi adj(3); adj[0]={1}; adj[1]={2}; adj[2]={0};
    cout << "hasCycleDirected(3-cycle) -> " << hasCycleDirected(3, adj) << " (Expected: 1)\n";
}

// DAG with a forward edge: 0->1, 1->2, 0->2 -- must NOT be flagged as a cycle
{
    vvi adj(3); adj[0]={1,2}; adj[1]={2}; adj[2]={};
    cout << "hasCycleDirected(DAG with forward edge) -> " << hasCycleDirected(3, adj) << " (Expected: 0)\n";
}

// The exact motivating example from the theory: a<-x<-u, u->a is a cycle where a != parent(u)
{
    // a=0, x=1, u=2:  0->1 (a->x), 1->2 (x->u), 2->0 (u->a).  parent(u=2) is x=1, but edge 2->0 closes a cycle with a=0.
    vvi adj(3); adj[0]={1}; adj[1]={2}; adj[2]={0};
    cout << "hasCycleDirected(cycle to non-parent ancestor) -> " << hasCycleDirected(3, adj) << " (Expected: 1, this is the same 3-cycle re-labeled to match the proof's example)\n";
}

// Cross edge case: 0->1, 2->1 -- 1 finishes before 2 is explored, edge 2->1 is a safe cross edge
{
    vvi adj(3); adj[0]={1}; adj[1]={}; adj[2]={1};
    cout << "hasCycleDirected(cross edge) -> " << hasCycleDirected(3, adj) << " (Expected: 0)\n";
}

// Self-loop: directed edge u->u is trivially a cycle (u is GRAY when its own self-edge is examined)
{
    vvi adj(1); adj[0] = {0};
    cout << "hasCycleDirected(self-loop) -> " << hasCycleDirected(1, adj) << " (Expected: 1)\n";
}

// Disconnected directed graph: one DAG component, one cyclic component
{
    vvi adj(5);
    adj[0]={1}; adj[1]={};              // DAG component
    adj[2]={3}; adj[3]={4}; adj[4]={2}; // cyclic component
    cout << "hasCycleDirected(disconnected, one cyclic) -> " << hasCycleDirected(5, adj) << " (Expected: 1)\n";
}


## 5.4 Bipartite Graph (BFS)

### State definition
$\text{color}[v] \in \{0, 1, -1\}$ where $-1$ = uncolored, and $0/1$ are the two partition classes.
A graph is **bipartite** iff a valid 2-coloring exists such that every edge connects vertices of
different colors.

### Invariant — layer parity
BFS from any root naturally partitions vertices into layers $\text{Layer}_0, \text{Layer}_1,
\ldots$ by distance (Subtopic 2.1). The bipartite coloring is exactly the **parity** of the layer:
$$
\text{color}[v] = \text{dist}[v] \bmod 2
$$
The invariant to maintain: every edge $(u,v)$ discovered during BFS must connect vertices of
*opposite* color — equivalently, opposite layer parity. If any edge instead connects two
same-colored (same-parity) vertices, the graph is not bipartite.

### Why it works — 2-coloring as a graph-coloring problem, formal correctness

**Claim:** a connected graph is bipartite iff BFS layer-parity coloring assigns different colors to
every edge's endpoints — equivalently, iff the graph has no odd-length cycle (matching the
Subtopic 1.5 terminology definition).

*Proof:* ($\Leftarrow$) if BFS's parity coloring is consistent (no edge connects same-parity
vertices), it is by construction a valid 2-coloring, so the graph is bipartite by definition.
($\Rightarrow$) if the graph is bipartite with true partition $V_1, V_2$, then every edge crosses
between $V_1$ and $V_2$; by induction on BFS layers starting from any root $r \in V_1$ (say),
every vertex at even distance from $r$ must be in $V_1$ and every vertex at odd distance in $V_2$ —
because each step of a shortest path alternates sides (an edge cannot stay within one side). Hence
layer parity exactly reconstructs a valid 2-coloring, so the BFS coloring succeeds.

**Failure signature:** BFS coloring fails to be consistent exactly when some edge closes an
odd-length cycle — an edge between two same-parity vertices means the path down to their common
ancestor plus this edge forms a cycle of odd total length (even + even + 1, or the general parity
argument), directly matching the odd-cycle characterization of non-bipartiteness.

### Boundary transitions table

| Decision point | Condition | Action |
|---|---|---|
| Neighbor uncolored | $\text{color}[w] = -1$ | assign $\text{color}[w] = 1 - \text{color}[u]$ (opposite), enqueue |
| Neighbor colored, opposite | $\text{color}[w] \neq \text{color}[u]$ | consistent, no action needed |
| Neighbor colored, same | $\text{color}[w] = \text{color}[u]$ | odd cycle detected — graph is not bipartite |
| Disconnected graph | some vertices unreached by BFS from one root | must restart BFS from every uncolored vertex — bipartiteness is checked per component |

### The delta
The non-obvious insight: bipartite-checking is **exactly** cycle-parity checking, not a separate
technique — the "same color as a neighbor" failure condition is structurally identical to 5.1's
"visited and not parent" cycle signal, except here the extra structure (assigning colors by parity)
lets you additionally certify *which* cycles are the problem (odd-length ones) rather than merely
detecting that a cycle exists at all.


In [ ]:
// Bipartite Graph (BFS): 2-coloring via layer parity
#include <iostream>
#include <vector>
#include <queue>
using namespace std;
#define vi vector<int>
#define vvi vector<vector<int>>

// bfsCheckBipartite: colors component containing src, returns false the instant a same-color edge is found.
bool bfsCheckBipartite(int src, vvi& adj, vi& color) {
    queue<int> q;
    color[src] = 0;               // arbitrary starting color for this component's root
    q.push(src);
    while (!q.empty()) {
        int u = q.front(); q.pop();
        for (int w : adj[u]) {
            if (color[w] == -1) {
                color[w] = 1 - color[u];     // opposite color = opposite layer parity
                q.push(w);
            } else if (color[w] == color[u]) {
                return false;                 // same color on an edge -> odd cycle -> not bipartite
            }
        }
    }
    return true;
}

bool isBipartite(int V, vvi& adj) {
    vi color(V, -1);
    for (int v = 0; v < V; v++) {
        if (color[v] == -1 && !bfsCheckBipartite(v, adj, color)) return false;  // check every component
    }
    return true;
}


In [ ]:
// Test cell: input -> actual (Expected: X)

// Even cycle (square): 0-1-2-3-0, bipartite
{
    vvi adj(4);
    int cyc[4]={0,1,2,3};
    for (int i=0;i<4;i++){int a=cyc[i], b=cyc[(i+1)%4]; adj[a].push_back(b); adj[b].push_back(a);}
    cout << "isBipartite(4-cycle, even) -> " << isBipartite(4, adj) << " (Expected: 1)\n";
}

// Odd cycle (triangle): 0-1-2-0, NOT bipartite
{
    vvi adj(3); adj[0]={1,2}; adj[1]={0,2}; adj[2]={0,1};
    cout << "isBipartite(triangle, odd cycle) -> " << isBipartite(3, adj) << " (Expected: 0)\n";
}

// Tree: always bipartite (no cycles at all)
{
    vvi adj(4); adj[0]={1}; adj[1]={0,2,3}; adj[2]={1}; adj[3]={1};
    cout << "isBipartite(tree) -> " << isBipartite(4, adj) << " (Expected: 1)\n";
}

// Single node
{
    vvi adj(1);
    cout << "isBipartite(single node) -> " << isBipartite(1, adj) << " (Expected: 1)\n";
}

// Star graph: bipartite (center vs leaves)
{
    vvi adj(5);
    for (int i=1;i<=4;i++){adj[0].push_back(i); adj[i].push_back(0);}
    cout << "isBipartite(star) -> " << isBipartite(5, adj) << " (Expected: 1)\n";
}

// Disconnected: one bipartite component, one odd-cycle component
{
    vvi adj(7);
    adj[0]={1}; adj[1]={0};                                  // bipartite edge component
    adj[2]={3,4}; adj[3]={2,4}; adj[4]={2,3}; adj[5]={};      // triangle {2,3,4} + isolated 5
    adj[6]={};
    cout << "isBipartite(disconnected, one odd cycle) -> " << isBipartite(7, adj) << " (Expected: 0)\n";
}


## 5.5 Bipartite Graph (DFS)

### State definition
Identical to 5.4: $\text{color}[v] \in \{0,1,-1\}$.

### Invariant
Same coloring rule, but colors are assigned along the DFS recursion order instead of BFS layers.
The invariant is the same consistency requirement ("no edge connects same-colored vertices"); DFS
just discovers vertices in a different order, so "layer parity" is replaced by "recursion-depth
parity along the DFS tree path from the root" — which is proven equivalent by the exact same
argument as 5.4 (any path from the root alternates sides in a true bipartition, regardless of
whether that path is a BFS shortest path or a DFS tree path).

### Why it works
Direct DFS analogue of 5.4's proof: coloring $v$ as $1 - \text{color}[\text{parent}]$ at discovery
time and checking every non-tree edge for a color clash is logically identical to the BFS version —
the *only* difference is traversal order, exactly the same relationship as 5.1 vs. 5.2 for cycle
detection.

### Boundary transitions table

Identical to 5.4's table, with "enqueue" replaced by "recurse."

### The delta
The non-obvious insight, closing this comparison: **bipartite checking, like undirected cycle
detection (5.1/5.2), is traversal-order-agnostic** — BFS and DFS are equally valid because the
underlying invariant (no edge between same-colored vertices) doesn't depend on *how* the coloring
was discovered, only on whether it's globally consistent. This is the same pattern as 5.1 vs 5.2,
and stands in contrast to 5.3's directed cycle detection, where DFS's GRAY-tracking is not
optional — BFS has no analogous "on the current recursion stack" concept to exploit.


In [ ]:
// Bipartite Graph (DFS): same 2-coloring rule, DFS recursion framing
#include <iostream>
#include <vector>
using namespace std;
#define vi vector<int>
#define vvi vector<vector<int>>

// dfsCheckBipartite: colors u, then recurses; returns false the instant a same-color edge is found.
bool dfsCheckBipartite(int u, vvi& adj, vi& color) {
    for (int w : adj[u]) {
        if (color[w] == -1) {
            color[w] = 1 - color[u];             // opposite color, assigned at discovery time
            if (!dfsCheckBipartite(w, adj, color)) return false;  // propagate failure up the recursion
        } else if (color[w] == color[u]) {
            return false;                          // same color on an edge -> odd cycle -> not bipartite
        }
    }
    return true;
}

bool isBipartiteDFS(int V, vvi& adj) {
    vi color(V, -1);
    for (int v = 0; v < V; v++) {
        if (color[v] == -1) {
            color[v] = 0;                          // seed this component's root
            if (!dfsCheckBipartite(v, adj, color)) return false;
        }
    }
    return true;
}


In [ ]:
// Test cell: input -> actual (Expected: X)

// Even cycle
{
    vvi adj(4);
    int cyc[4]={0,1,2,3};
    for (int i=0;i<4;i++){int a=cyc[i], b=cyc[(i+1)%4]; adj[a].push_back(b); adj[b].push_back(a);}
    cout << "isBipartiteDFS(4-cycle) -> " << isBipartiteDFS(4, adj) << " (Expected: 1)\n";
}

// Odd cycle
{
    vvi adj(3); adj[0]={1,2}; adj[1]={0,2}; adj[2]={0,1};
    cout << "isBipartiteDFS(triangle) -> " << isBipartiteDFS(3, adj) << " (Expected: 0)\n";
}

// Single node
{
    vvi adj(1);
    cout << "isBipartiteDFS(single node) -> " << isBipartiteDFS(1, adj) << " (Expected: 1)\n";
}

// Fully connected K4 -- every pair adjacent, definitely not bipartite (contains triangles)
{
    vvi adj(4);
    for (int i=0;i<4;i++) for (int j=0;j<4;j++) if (i!=j) adj[i].push_back(j);
    cout << "isBipartiteDFS(K4) -> " << isBipartiteDFS(4, adj) << " (Expected: 0)\n";
}

// Cross-check against BFS result (5.4) on the same disconnected mixed graph
{
    vvi adj(7);
    adj[0]={1}; adj[1]={0};
    adj[2]={3,4}; adj[3]={2,4}; adj[4]={2,3}; adj[5]={};
    adj[6]={};
    cout << "isBipartiteDFS(disconnected, one odd cycle) -> " << isBipartiteDFS(7, adj) << " (Expected: 0, matches BFS result from 5.4)\n";
}

// Bipartite complete bipartite graph K(2,3): every left vertex connects to every right vertex
{
    vvi adj(5);
    // left = {0,1}, right = {2,3,4}
    for (int l=0;l<2;l++) for (int r=2;r<5;r++) { adj[l].push_back(r); adj[r].push_back(l); }
    cout << "isBipartiteDFS(K(2,3)) -> " << isBipartiteDFS(5, adj) << " (Expected: 1)\n";
}


## 5.6 Comparison Table — Which Cycle Detection Uses Which Technique, and Why

| Variant | Technique | Why this technique (not the other) |
|---|---|---|
| Undirected cycle detection | Parent tracking (BFS **or** DFS, either works) | Every non-tree edge in an undirected graph is a back edge to *some* ancestor (Subtopic 2.2 proof: no forward/cross edges exist), and every edge is bidirectional, so "visited and not the edge I just arrived on" is a complete and correct test. Traversal order is irrelevant to this argument. |
| Directed cycle detection | Three-color (WHITE/GRAY/BLACK), **DFS only** | Directed edges can point to already-visited vertices that are *not* the parent and *not* an active ancestor (forward/cross edges), which parent-exclusion cannot distinguish from a real cycle-closing back edge. Only the GRAY state (recursion-stack membership) correctly isolates "active ancestor" from "finished, unrelated vertex." BFS has no natural analogue of "on the current recursion stack," so this genuinely requires DFS. |
| Bipartite check | Layer/recursion parity coloring (BFS **or** DFS, either works) | Structurally the same "traversal-order-agnostic" situation as undirected cycle detection — consistency of a 2-coloring doesn't depend on discovery order, only on whether some edge ends up connecting same-colored vertices, which is detectable identically via BFS layers or DFS recursion. |

### The one-sentence summary
**Bidirectional edges make traversal order irrelevant** (undirected cycle detection, bipartite
check — both work with either BFS or DFS via parent/parity tracking); **directed edges make
traversal order essential** (directed cycle detection needs DFS's recursion-stack state
specifically, because that is the only structure that distinguishes "currently on my path back to
the root" from "already fully explored, possibly still reachable from me by coincidence").


In [ ]:
// ================= UNIFIED MENTAL MODEL — Subtopic 5 =================
#include <iostream>
#include <vector>
#include <queue>
using namespace std;
#define vi vector<int>
#define vvi vector<vector<int>>

enum Color3 { W, G, B };

// --- Template: undirected cycle detection (parent-based, works via BFS or DFS) ---
bool undirectedCycleTemplate(int u, int parent, vvi& adj, vector<bool>& visited) {
    visited[u] = true;
    for (int w : adj[u]) {
        if (!visited[w]) { if (undirectedCycleTemplate(w, u, adj, visited)) return true; }
        else if (w != parent) return true;
    }
    return false;
}

// --- Template: directed cycle detection (three-color, DFS required) ---
bool directedCycleTemplate(int u, vvi& adj, vector<Color3>& color) {
    color[u] = G;
    for (int v : adj[u]) {
        if (color[v] == W) { if (directedCycleTemplate(v, adj, color)) return true; }
        else if (color[v] == G) return true;
    }
    color[u] = B;
    return false;
}

// --- Template: bipartite check (parity-based, works via BFS or DFS) ---
bool bipartiteTemplate(int src, vvi& adj, vi& color) {
    queue<int> q; color[src] = 0; q.push(src);
    while (!q.empty()) {
        int u = q.front(); q.pop();
        for (int w : adj[u]) {
            if (color[w] == -1) { color[w] = 1 - color[u]; q.push(w); }
            else if (color[w] == color[u]) return false;
        }
    }
    return true;
}

int main() {
    vvi adj(3); adj[0]={1}; adj[1]={2}; adj[2]={0};   // directed 3-cycle
    vector<Color3> color(3, W);
    cout << "Unified template sanity check: directed cycle -> " << directedCycleTemplate(0, adj, color) << " (Expected: 1)\n";
    return 0;
}


### Decision Tree — Subtopic 5

```
Is the graph DIRECTED or UNDIRECTED?
│
├── UNDIRECTED
│    │
│    Are you checking for a cycle, or checking bipartiteness?
│    ├── Cycle       → parent tracking, BFS or DFS (either works, 5.1/5.2)
│    └── Bipartite   → layer/recursion parity coloring, BFS or DFS (either works, 5.4/5.5)
│    (traversal order doesn't matter in either case -- undirected graphs have no forward/cross edges)
│
└── DIRECTED
     │
     Checking for a cycle?
     ├── YES → three-color WHITE/GRAY/BLACK, DFS ONLY (5.3)
     │         (parent-tracking is NOT sufficient -- forward/cross edges break it; BFS has no
     │          recursion-stack analogue to exploit)
     └── NO, checking something else on a directed graph (e.g. topological order) → see Subtopic 7
```


### Complexity Summary — Subtopic 5

| Algorithm | Time | Space | When to use | Key invariant | Failure mode if misused |
|---|---|---|---|---|---|
| Undirected cycle detection (BFS) | $O(V+E)$ | $O(V)$ | Undirected graph, need cycle existence | visited-and-not-parent signals a genuine back edge | none specific — BFS/DFS interchangeable here |
| Undirected cycle detection (DFS) | $O(V+E)$ | $O(V)$ recursion | Same as above, DFS-preferred codebases | identical to BFS version, via recursion-stack ancestry instead of layers | none specific |
| Directed cycle detection (DFS, 3-color) | $O(V+E)$ | $O(V)$ recursion + color array | Directed graph, need cycle existence | GRAY $\iff$ active ancestor on current recursion stack | using parent-exclusion instead of GRAY silently misses cycles to non-parent ancestors, or misflags safe cross/forward edges |
| Bipartite check (BFS) | $O(V+E)$ | $O(V)$ | 2-coloring / odd-cycle detection | layer parity = valid coloring iff no odd cycle exists | none specific — BFS/DFS interchangeable here |
| Bipartite check (DFS) | $O(V+E)$ | $O(V)$ recursion | Same as above, DFS-preferred codebases | recursion-depth parity along DFS tree path, same consistency rule | none specific |
